# Segger Checkpoint and Segmentation Workflows


## Requirements

- Segger installed (with GPU dependencies if running segmentation)
- A dataset in raw platform format or SpatialData Zarr
- A Lightning checkpoint (.ckpt) if using checkpoint modes


## Scenario 1: Standard segmentation (train + predict)

```bash
segger segment -i /path/to/data -o /path/to/out
```

Notes:
- This trains a new model and then runs prediction.
- Use `--output-format` to write merged, spatialdata, or anndata outputs.


## Scenario 2: Predict-only from a checkpoint

```bash
segger segment -i /path/to/data -o /path/to/out \
  --checkpoint-path /path/to/model.ckpt \
  --checkpoint-mode predict
```

Notes:
- Skips training and only runs prediction.
- Uses the checkpoint model weights as-is.


## Scenario 3: Resume training from a checkpoint

```bash
segger segment -i /path/to/data -o /path/to/out \
  --checkpoint-path /path/to/model.ckpt \
  --checkpoint-mode resume \
  --n-epochs 20
```

Notes:
- Restores optimizer state and resumes training.
- Requires identical gene vocabulary (`vocab_mode` is strict).


## Scenario 4: Finetune from a checkpoint

```bash
segger segment -i /path/to/data -o /path/to/out \
  --checkpoint-path /path/to/model.ckpt \
  --checkpoint-mode finetune \
  --learning-rate 1e-4 \
  --n-epochs 5
```

Notes:
- Loads weights but starts a fresh optimizer.
- Best practice: small learning rate and short finetune.


## Scenario 5: Vocab overlap finetune

If your current gene list differs from the checkpoint, use overlap mode.

```bash
segger segment -i /path/to/data -o /path/to/out \
  --checkpoint-path /path/to/model.ckpt \
  --checkpoint-mode finetune \
  --vocab-mode overlap \
  --checkpoint-vocab /path/to/checkpoint_genes.txt
```

Notes:
- `checkpoint_genes.txt` is one gene per line in checkpoint order.
- Overlapping genes reuse checkpoint weights; new genes are initialized.


## Scenario 6: Freeze or update gene embeddings

```bash
# Freeze gene embeddings during finetune
segger segment -i /path/to/data -o /path/to/out \
  --checkpoint-path /path/to/model.ckpt \
  --checkpoint-mode finetune \
  --update-gene-embedding false

# Update gene embeddings during finetune
segger segment -i /path/to/data -o /path/to/out \
  --checkpoint-path /path/to/model.ckpt \
  --checkpoint-mode finetune \
  --update-gene-embedding true
```

Notes:
- Updating embeddings can help new genes adapt in overlap mode.
- Freezing can stabilize training when data is limited.


## Scenario 7: Export gene embeddings

```bash
segger segment -i /path/to/data -o /path/to/out \
  --export-gene-embeddings \
  --gene-embeddings-filename gene_embeddings.parquet
```

Notes:
- Writes a parquet with `feature_name` plus embedding columns `emb_0..emb_n`.
- Useful for downstream analysis or reuse.


## Scenario 8: Combine checkpointing with output formats

```bash
segger segment -i /path/to/data -o /path/to/out \
  --checkpoint-path /path/to/model.ckpt \
  --checkpoint-mode predict \
  --output-format all
```

Notes:
- Produces segmentation parquet plus merged, spatialdata, and anndata outputs.


## Summary

- Use `checkpoint-mode predict` for inference-only.
- Use `checkpoint-mode finetune` for transfer learning.
- Use `vocab-mode overlap` when gene lists differ and provide `checkpoint-vocab`.
- Export gene embeddings when you need reusable gene features.

---

## Python API for Checkpoint Inspection

You can programmatically inspect checkpoints before using them.

### Load and inspect checkpoint metadata

In [ ]:
import torch
from pathlib import Path

# Load checkpoint and inspect metadata
checkpoint_path = Path("model.ckpt")  # Replace with your checkpoint path

# Check if checkpoint exists before loading
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    
    # Check for segger_metadata (new format)
    if "segger_metadata" in checkpoint:
        meta = checkpoint["segger_metadata"]
        print(f"Segger version: {meta.get('segger_version', 'unknown')}")
        print(f"Gene count: {meta.get('n_genes', 'unknown')}")
        print(f"Embedding dim: {meta.get('embedding_dim', 'unknown')}")
        print(f"Saved at: {meta.get('saved_at', 'unknown')}")
    else:
        # Fallback to hyper_parameters (older checkpoints)
        hparams = checkpoint.get("hyper_parameters", {})
        print(f"Gene count: {hparams.get('n_genes', 'unknown')}")
        print(f"Gene names available: {'gene_names' in hparams}")
else:
    print(f"Checkpoint not found: {checkpoint_path}")

### Extract gene names for vocab file

In [ ]:
# Extract gene names from checkpoint (useful for vocab overlap)
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    
    # Try segger_metadata first, then hyper_parameters
    gene_names = None
    if "segger_metadata" in checkpoint:
        gene_names = checkpoint["segger_metadata"].get("gene_names", [])
    elif "hyper_parameters" in checkpoint:
        gene_names = checkpoint["hyper_parameters"].get("gene_names", [])
    
    if gene_names:
        # Write to file for --checkpoint-vocab
        vocab_file = Path("checkpoint_genes.txt")
        with open(vocab_file, "w") as f:
            f.write("\n".join(gene_names))
        print(f"Wrote {len(gene_names)} gene names to {vocab_file}")
    else:
        print("No gene names found in checkpoint metadata.")

### Using the CheckpointMetadata class

In [ ]:
# Use the CheckpointMetadata class for structured access
from segger.models import CheckpointMetadata

# Load metadata from checkpoint (requires Segger v0.2.0+ checkpoints)
if checkpoint_path.exists():
    try:
        metadata = CheckpointMetadata.from_checkpoint(checkpoint_path)
        if metadata:
            print(f"Checkpoint: {metadata}")
            print(f"Gene count: {metadata.n_genes}")
            
            # Check compatibility with another checkpoint
            # other_meta = CheckpointMetadata.from_checkpoint("other_model.ckpt")
            # warnings = metadata.validate_against(other_meta)
            # for w in warnings:
            #     print(f"Warning: {w}")
            
            # Calculate gene overlap with new data
            new_gene_names = ["Gene1", "Gene2", "Gene3"]  # Replace with your genes
            overlap = metadata.get_gene_overlap(new_gene_names)
            print(f"Overlap: {overlap['overlap_count']}/{metadata.n_genes} genes "
                  f"({overlap['overlap_ratio']:.1%})")
    except Exception as e:
        print(f"Error loading checkpoint metadata: {e}")

---

## Verifying Checkpoint Loading

After running segmentation with a checkpoint, verify the loading was successful.

### Check overlap count in logs

When using `--vocab-mode overlap`, the logs will show:

```
INFO: Remapped 18,500/20,000 genes from checkpoint (92.5% overlap)
```

### Compare gene embeddings

In [ ]:
import polars as pl
from pathlib import Path

# Compare embeddings before and after checkpoint loading
old_embeddings_path = Path("checkpoint_embeddings.parquet")
new_embeddings_path = Path("output/gene_embeddings.parquet")

if old_embeddings_path.exists() and new_embeddings_path.exists():
    old = pl.read_parquet(old_embeddings_path)
    new = pl.read_parquet(new_embeddings_path)
    
    # Find shared genes
    old_genes = set(old["feature_name"].to_list())
    new_genes = set(new["feature_name"].to_list())
    shared = old_genes & new_genes
    print(f"Shared genes: {len(shared)}")
    
    # For shared genes, embeddings should be identical
    # (unless update_gene_embedding was True)
    if shared:
        sample_gene = list(shared)[0]
        old_emb = old.filter(pl.col("feature_name") == sample_gene)
        new_emb = new.filter(pl.col("feature_name") == sample_gene)
        print(f"Sample gene '{sample_gene}' embeddings match: ", end="")
        # Compare embedding columns
        emb_cols = [c for c in old.columns if c.startswith("emb_")]
        old_vals = old_emb.select(emb_cols).to_numpy().flatten()
        new_vals = new_emb.select(emb_cols).to_numpy().flatten()
        print(abs(old_vals - new_vals).max() < 1e-5)
else:
    print("Embedding files not found. Export with --export-gene-embeddings.")

---

## Troubleshooting

### Error: "Checkpoint gene count does not match current data"

**Cause:** Different gene filtering between datasets.

**Solution:** Use `--vocab-mode overlap` with `--checkpoint-vocab`:

```bash
segger segment -i data/ -o output/ \
  --checkpoint-path model.ckpt \
  --checkpoint-mode finetune \
  --vocab-mode overlap \
  --checkpoint-vocab checkpoint_genes.txt
```

### Error: "vocab_mode='overlap' requires checkpoint gene names"

**Cause:** Old checkpoint lacks gene_names in hparams/metadata.

**Solution:** Create a gene vocab file manually using the Python API above, or retrain the checkpoint with a newer Segger version.

### Warning: "LR scheduler state may not match"

**Cause:** Different batch size or dataset size on resume, particularly with OneCycleLR which tracks total steps.

**Solution:** Consider using `--checkpoint-mode finetune` instead, which starts a fresh optimizer and scheduler.

### Warning: "Checkpoint embedding dimension differs from current model"

**Cause:** The checkpoint was trained with a different `--node-representation-dim` value.

**Solution:** Use the same `--node-representation-dim` as the checkpoint, or retrain from scratch.

### Low transcript assignment rate after checkpoint loading

**Cause:** Low gene overlap between checkpoint and new data.

**Solutions:**
1. Check overlap with the Python API above
2. Use `--update-gene-embedding true` to allow new genes to adapt
3. Train longer with finetune mode
4. Consider training from scratch if overlap < 50%

### Checkpoint file too large

**Cause:** Gene names are stored in hparams, which can be large for many genes.

**Solution:** This is expected behavior for reproducibility. The checkpoint size is proportional to vocabulary size. For inference-only deployment, you can strip gene_names from the checkpoint manually.